# Practice 25 — pandas + NumPy
Work through each task in order. Try to solve it yourself before looking anything up!

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — Basics

### Task 1: titanic — passenger profiling

Load `sns.load_dataset('titanic')`.

Five questions — no hints:

- Which `(sex, pclass)` combination has the highest survival rate?
- Mean age per passenger class, ignoring nulls. Use NumPy.
- The `name` column contains entries like `'Braund, Mr. Owen Harris'`. Use `.str.extract()` to pull out the title (e.g. `'Mr'`, `'Mrs'`, `'Miss'`, `'Master'`). Which title has the highest survival rate?
- Add `family_size = sibsp + parch + 1`. Among passengers with `family_size >= 4`, what fraction survived? Use NumPy.
- Is there a correlation between `fare` and `family_size`? Use NumPy. Does the sign match your intuition?

In [17]:
# Your code here

titanic = sns.load_dataset('titanic')
titanic.groupby(['sex','pclass']).apply(lambda x: (x['survived'].sum())/(x['survived'].agg('count'))).idxmax()
titanic.groupby('pclass')['age'].agg(np.nanmean)
#titanic['title'] = titanic['name'].str.extract(r'Mr|Mrs|Miss|Master')
#titanic['name'].str.extract(r'Mr|Mrs|Miss|Master')
#'name' is not one of the columns

titanic['family_size'] = titanic['sibsp']+titanic['parch']+1
f = titanic[titanic['family_size']>=4]
np.mean(f['survived'])


np.corrcoef(titanic['family_size'], titanic['fare'])

/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_51921/1444518323.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  titanic.groupby(['sex','pclass']).apply(lambda x: (x['survived'].sum())/(x['survived'].agg('count'))).idxmax()
/var/folders/3r/5sttq01d46zg8zxyw17j5nbw0000gn/T/ipykernel_51921/1444518323.py:5: FutureWarning: The provided callable <function nanmean at 0x7fc5ba624b80> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  titanic.groupby('pclass')['age'].agg(np.nanmean)


array([[1.        , 0.21713841],
       [0.21713841, 1.        ]])

---
## Level 2 — Transformations

### Task 2: Exam scores — z-score reshape

A wide DataFrame of student exam scores has been created for you.

1. Stack to long format. Name the value column `'score'`. Store as `scores_long`.
2. Add `z_score`: standardize each score *within its subject* — subtract the subject mean and divide by the subject std. Do this without `.apply()` — use `transform` twice.
3. Add `subject_rank`: rank each student within their subject by `score`, highest first.
4. Which student has the highest mean `z_score` across all subjects?
5. Unstack `z_score` back to wide format (students as rows, subjects as columns).
6. Use `.xs()` to extract the full z-score profile for `'Physics'`.

In [37]:
# Starter data — don't change this
np.random.seed(19)
students = ['Ava', 'Ben', 'Cleo', 'Dom', 'Eve', 'Finn']
subjects = ['Math', 'Physics', 'History', 'English']
scores_wide = pd.DataFrame(
    np.random.randint(50, 100, size=(len(students), len(subjects))),
    index=students,
    columns=subjects
)
scores_wide.index.name = 'student'

# Your code here

scores_long = scores_wide.stack().to_frame('score')
scores_long['zscore'] = (scores_long['score']-scores_long.groupby(level=1)['score'].transform('mean'))/(scores_long.groupby(level=1)['score'].transform('std'))
scores_long['subject_rank'] = scores_long.groupby(level=1)['score'].rank(ascending=False)
scores_long.groupby('student')['zscore'].mean().idxmax()

w = scores_long['zscore'].unstack()
scores_long['zscore'].xs('Physics', level=1)

student
Ava     1.252492
Ben    -1.079735
Cleo    0.475083
Dom    -0.302326
Eve     0.799004
Finn   -1.144519
Name: zscore, dtype: float64

### Task 3: .pipe() — inventory pipeline

Write all three functions yourself and chain them with `.pipe()`:

- `add_value(df)`: `total_value = units_in_stock * unit_cost`
- `add_reorder_flag(df)`: `needs_reorder = True` if `units_in_stock < reorder_threshold`
- `categorize_margin(df)`: add `margin_tier` — `'High'` if `margin_pct > 0.4`, `'Mid'` if `> 0.2`, else `'Low'` — using `.apply()`

After the pipeline:
- How many items need reorder per `category`?
- Which `category` has the highest mean `total_value`? Use NumPy.
- Stack the result so each row is `(sku, attribute)`. Use `.xs()` to extract just the `total_value` column across all SKUs.

In [ ]:
# Starter data — don't change this
np.random.seed(33)
inventory = pd.DataFrame({
    'sku':               [f'SK{i:02d}' for i in range(1, 10)],
    'category':          np.random.choice(['Electronics', 'Apparel', 'Home'], size=9),
    'units_in_stock':    np.random.randint(0, 200, size=9),
    'unit_cost':         np.round(np.random.uniform(5, 150, size=9), 2),
    'margin_pct':        np.round(np.random.uniform(0.05, 0.65, size=9), 2),
    'reorder_threshold': np.random.randint(10, 60, size=9),
})

# Your code here

def add_value(df):
    df['total_value'] = df['units_in_stock']*df['unit_cost']
    return df

def add_reorder_flag(df):
    df['needs_reorder'] = df['units_in_stock'] < df['reorder_threshold']
    return df

def categorize_margin(df):
    df['margin_tier'] = df['margin_pct'].apply(lambda x: 'High' if x>0.4
                                               else 'Mid' if x >0.2
                                               else 'Low')
    return df

res = (
    inventory
    .pipe(add_value)
    .pipe(add_reorder_flag)
    .pipe(categorize_margin)
)

res.groupby('category')['needs_reorder'].sum()
m = res.groupby('category')['total_value'].mean()
m.index[np.argmax(m)]


### there is no 'attribute' in the df

,sku,category,units_in_stock,unit_cost,margin_pct,reorder_threshold,total_value,needs_reorder,margin_tier
0,SK01,Electronics,170,144.63,0.51,40,24587.10,False,High
1,SK02,Electronics,45,72.18,0.17,36,3248.10,False,Low
2,SK03,Home,35,20.54,0.20,21,718.90,False,Low
3,SK04,Home,33,113.14,0.30,11,3733.62,False,Mid
4,SK05,Apparel,77,144.80,0.39,59,11149.60,False,Mid
5,SK06,Apparel,31,19.23,0.12,25,596.13,False,Low
6,SK07,Home,172,30.77,0.12,46,5292.44,False,Low
7,SK08,Apparel,122,57.71,0.60,43,7040.62,False,High
8,SK09,Home,140,125.47,0.45,21,17565.80,False,High


---
## Level 3 — Aggregation

### Task 4: Employee compensation — merge, pipeline, MultiIndex

Two DataFrames: `employees` and `departments`. No step-by-step list.

Your job:

1. Join them on `dept_id`.
2. Design and build a `.pipe()` chain with exactly **3 functions of your own choosing**. Requirements:
   - At least one function must use `transform` to compute a department-level metric
   - At least one function must classify employees using `.apply()`
   - At least one function must flag something based on two conditions
3. Set a `(dept_name, emp_id)` MultiIndex on the result. Sort it.
4. Answer these questions from the final DataFrame:
   - Which department has the highest mean salary?
   - Retrieve all rows for the `'Engineering'` department.
   - Which `(dept_name, emp_id)` has the highest salary? Find it without `.idxmax()` — use NumPy.

In [54]:
# Starter data — don't change this
np.random.seed(51)
employees = pd.DataFrame({
    'emp_id':      [f'E{i:02d}' for i in range(1, 13)],
    'dept_id':     np.random.choice(['D01', 'D02', 'D03', 'D04'], size=12),
    'salary':      np.random.randint(55000, 145000, size=12),
    'years_exp':   np.random.randint(1, 18, size=12),
    'performance': np.random.randint(1, 6, size=12),
})

departments = pd.DataFrame({
    'dept_id':   ['D01', 'D02', 'D03', 'D04'],
    'dept_name': ['Engineering', 'Product', 'Design', 'Marketing'],
    'budget':    [800000, 500000, 300000, 400000],
})

# Your code here

ij = pd.merge(
    employees, 
    departments, 
    on='dept_id'
)

def dept_mean_salary(df):
    df['dept_mean_salary'] = df.groupby('dept_name')['salary'].transform('mean')
    return df
def add_emp_senoir(df):
    df['senior_flag'] = df['years_exp'].apply(lambda x: 'Senoir' if x>=10
                                              else 'Junior')
    return df
def add_dept_potency(df):
    df['potent_dept_flag'] = df.apply(lambda row: 'Potent' if row['budget']> 400000 and row['dept_mean_salary']>70000
                                      else "not Potent", axis = 1)
    return df


res = (
    ij
    .pipe(dept_mean_salary)
    .pipe(add_emp_senoir)
    .pipe(add_dept_potency)
)

res = res.set_index(['dept_name','emp_id']).sort_index()

res.groupby(level='dept_name',observed=True)['salary'].mean().idxmax()

res.xs('Engineering', level='dept_name')
s = res.groupby(level= ['dept_name','emp_id'])['salary'].mean()
s.index[np.argmax(s)]

('Engineering', 'E07')

---
## Level 4 — Real-world

### Task 5: iris — measurement separability

Load `sns.load_dataset('iris')`.

Four questions. No scaffolding.

1. Which species has the most consistent measurements across all numeric columns? Define consistency yourself — show the numbers.
2. Which single measurement **best separates** the three species? Think about it this way: a good separating measurement has high variance *between* species means and low variance *within* each species. Use NumPy to compute a ratio and compare across all four measurements.
3. Build a `.pipe()` chain with at least two functions. The pipeline should produce at least two new columns. End with a summary that reveals something non-obvious.
4. Create a reshaped view — pivot, stack, or unstack — that compares all four measurements across species side by side. Use `.xs()` somewhere.

In [ ]:
# Your code here

iris = sns.load_dataset('iris')
iris_numeric = iris.select_dtypes(include='number')
iris_numeric
iris['consistency'] = iris_numeric.std(axis = 1)
iris.groupby('species')['consistency'].mean().idxmin()

cols = iris.select_dtypes(include='number').columns
feature_ratio = {}
for f in cols:
    indi = iris.groupby('species')[f].agg('mean')
    within = iris.groupby('species')[f].agg('std').mean()
    between = indi.std()
    ratio = between/within
    feature_ratio[f] = ratio

max(feature_ratio, key=feature_ratio.get)



KeyError: ''

In [83]:
iris_view = iris.groupby('species')[cols].mean().stack().unstack('species')
iris_view

species,setosa,versicolor,virginica
sepal_length,5.006000,5.936000,6.58800
sepal_width,3.428000,2.770000,2.97400
petal_length,1.462000,4.260000,5.55200
petal_width,0.246000,1.326000,2.02600
consistency,2.110321,1.986714,2.14626


In [90]:
iris_view = iris.groupby('species')[cols].mean().stack()
iris_view.index

MultiIndex([(    'setosa', 'sepal_length'),
            (    'setosa',  'sepal_width'),
            (    'setosa', 'petal_length'),
            (    'setosa',  'petal_width'),
            (    'setosa',  'consistency'),
            ('versicolor', 'sepal_length'),
            ('versicolor',  'sepal_width'),
            ('versicolor', 'petal_length'),
            ('versicolor',  'petal_width'),
            ('versicolor',  'consistency'),
            ( 'virginica', 'sepal_length'),
            ( 'virginica',  'sepal_width'),
            ( 'virginica', 'petal_length'),
            ( 'virginica',  'petal_width'),
            ( 'virginica',  'consistency')],
           names=['species', None])